# Chatbot Rekomendasi Wisata Yogyakarta - OPTIMIZED VERSION

**Problem**: 88 intents dengan ~25 patterns per intent (MANY CLASSES, FEW SAMPLES)

**Solutions**:
- Gunakan data FULL (bukan augmented yang repetitive)
- Strategi khusus untuk many-class classification
- Data augmentation yang smart (bukan repetisi)
- Test set lebih besar (30%) untuk better evaluation
- Regularization yang lebih kuat


## 1. Import Libraries


In [ ]:
import json
import numpy as np
import pandas as pd
import random
import warnings
warnings.filterwarnings('ignore')

# Text preprocessing
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Feature extraction
from sklearn.feature_extraction.text import TfidfVectorizer

# Model selection and evaluation
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

# Models
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

# Calibration for probability
from sklearn.calibration import CalibratedClassifierCV

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Save model
import pickle

# NLTK downloads
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Random seed
np.random.seed(42)
random.seed(42)

print('Libraries loaded successfully!')


## 2. Load and Analyze Data


In [ ]:
# Load CLEANED data (merged time-greetings, removed duplicates)
with open('data/intents_diy_cleaned_aggressive.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

print(f"Total intents: {len(data['intents'])}")

# Extract patterns
patterns = []
tags = []
responses_dict = {}

for intent in data['intents']:
    tag = intent['tag']
    responses_dict[tag] = intent['responses']
    for pattern in intent['patterns']:
        patterns.append(pattern)
        tags.append(tag)

# Create DataFrame
df = pd.DataFrame({'pattern': patterns, 'tag': tags})

print(f"\nDataset shape: {df.shape}")
print(f"Unique tags: {df['tag'].nunique()}")
print(f"\nTag distribution summary:")
print(df['tag'].value_counts().describe())



In [ ]:
# Visualize class distribution
plt.figure(figsize=(16, 6))
tag_counts = df['tag'].value_counts()
plt.bar(range(len(tag_counts)), tag_counts.values, color='steelblue', alpha=0.7)
plt.xlabel('Intent', fontsize=12)
plt.ylabel('Number of Patterns', fontsize=12)
plt.title(f'Class Distribution: {len(tag_counts)} Intents', fontsize=14, fontweight='bold')
plt.axhline(y=tag_counts.mean(), color='red', linestyle='--', label=f'Mean: {tag_counts.mean():.0f}')
plt.axhline(y=tag_counts.median(), color='green', linestyle='--', label=f'Median: {tag_counts.median():.0f}')
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"\n⚠️ CHALLENGE: {len(tag_counts)} classes with avg {tag_counts.mean():.0f} samples each")
print(f"This is a MANY-CLASS, FEW-SAMPLES problem!")


## 3. Text Preprocessing


In [ ]:
# Initialize stemmer
factory = StemmerFactory()
stemmer = factory.create_stemmer()
stop_words = set(stopwords.words('indonesian'))

def preprocess_text(text):
    # Lowercase
    text = text.lower()
    # Remove special chars, keep alphabets and spaces
    text = re.sub(r'[^a-z\s]', '', text)
    # Tokenization
    tokens = word_tokenize(text)
    # Remove stopwords and stem
    tokens = [stemmer.stem(word) for word in tokens if word not in stop_words and len(word) > 2]
    return ' '.join(tokens)

# Test
sample = "Halo, saya ingin wisata di Bantul!"
print(f"Original: {sample}")
print(f"Processed: {preprocess_text(sample)}")

# Apply to all
print("\nPreprocessing all patterns...")
df['processed_pattern'] = df['pattern'].apply(preprocess_text)
df = df[df['processed_pattern'].str.strip() != '']
print(f"Dataset after preprocessing: {df.shape}")


## 4. Feature Extraction

**Strategy for many-class problem**:
- Higher max_features (more vocabulary for 88 classes)
- Character n-grams for robustness
- Lower min_df to keep rare but important terms


In [ ]:
# TF-IDF with optimization for many classes
tfidf_vectorizer = TfidfVectorizer(
    max_features=2000,      # Increased for 88 classes
    ngram_range=(1, 3),     # Unigrams, bigrams, trigrams
    min_df=1,               # Keep all terms (small dataset)
    max_df=0.9,             # Remove very common terms
    sublinear_tf=True,      # Use log scaling
    analyzer='word'
)

X = tfidf_vectorizer.fit_transform(df['processed_pattern'])
y = df['tag']

print(f"Feature matrix: {X.shape}")
print(f"Features: {len(tfidf_vectorizer.get_feature_names_out())}")
print(f"Classes: {len(y.unique())}")
print(f"Samples per class (avg): {len(df) / len(y.unique()):.1f}")


## 5. Train-Test Split

**Important**: Use 30% test for better evaluation with small data


In [ ]:
# 70/30 split (more test data for reliable evaluation)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training: {X_train.shape[0]} samples")
print(f"Testing: {X_test.shape[0]} samples")
print(f"Avg samples per class in training: {X_train.shape[0] / len(y.unique()):.1f}")
print(f"Avg samples per class in testing: {X_test.shape[0] / len(y.unique()):.1f}")

if X_train.shape[0] / len(y.unique()) < 20:
    print("\n⚠️ WARNING: Less than 20 training samples per class!")
    print("   This is VERY challenging. Model will struggle with rare classes.")


## 6. Model Training - Optimized for Many Classes

**Best models for many-class, few-sample**:
1. Linear SVM (best for text, handles many classes well)
2. Logistic Regression with L2 (good baseline)
3. Multinomial Naive Bayes (fast, probabilistic)

**NOT using**: Random Forest, Neural Networks (need more data)


### 6.1 Linear SVM (Recommended)


In [ ]:
print("Training Linear SVM...")

# Linear SVM is best for text classification with many classes
svm_model = LinearSVC(
    C=0.1,                  # Strong regularization for small data
    max_iter=5000,
    class_weight='balanced', # Handle any imbalance
    random_state=42,
    dual=False              # Faster for n_samples > n_features
)

svm_model.fit(X_train, y_train)

# Calibrate for probability estimates
svm_calibrated = CalibratedClassifierCV(svm_model, cv=3)
svm_calibrated.fit(X_train, y_train)

# Predictions
y_pred_svm_train = svm_model.predict(X_train)
y_pred_svm = svm_model.predict(X_test)

# Evaluation
train_acc = accuracy_score(y_train, y_pred_svm_train)
test_acc = accuracy_score(y_test, y_pred_svm)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred_svm, average='weighted', zero_division=0)

print(f"\nLinear SVM Results:")
print(f"  Training Accuracy: {train_acc:.4f}")
print(f"  Test Accuracy:     {test_acc:.4f}")
print(f"  Gap:               {train_acc - test_acc:.4f}")
print(f"  Precision:         {precision:.4f}")
print(f"  Recall:            {recall:.4f}")
print(f"  F1-Score:          {f1:.4f}")

# Cross-validation
cv_scores = cross_val_score(svm_model, X_train, y_train, cv=min(5, X_train.shape[0]//len(y.unique())), scoring='accuracy')
print(f"  CV Accuracy:       {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")


### 6.2 Logistic Regression


In [ ]:
print("Training Logistic Regression...")

lr_model = LogisticRegression(
    C=0.5,                  # Regularization
    max_iter=1000,
    class_weight='balanced',
    solver='lbfgs',
    multi_class='multinomial',
    random_state=42,
    n_jobs=-1
)

lr_model.fit(X_train, y_train)

# Predictions
y_pred_lr_train = lr_model.predict(X_train)
y_pred_lr = lr_model.predict(X_test)

# Evaluation
train_acc_lr = accuracy_score(y_train, y_pred_lr_train)
test_acc_lr = accuracy_score(y_test, y_pred_lr)
precision_lr, recall_lr, f1_lr, _ = precision_recall_fscore_support(y_test, y_pred_lr, average='weighted', zero_division=0)

print(f"\nLogistic Regression Results:")
print(f"  Training Accuracy: {train_acc_lr:.4f}")
print(f"  Test Accuracy:     {test_acc_lr:.4f}")
print(f"  Gap:               {train_acc_lr - test_acc_lr:.4f}")
print(f"  Precision:         {precision_lr:.4f}")
print(f"  Recall:            {recall_lr:.4f}")
print(f"  F1-Score:          {f1_lr:.4f}")

# Cross-validation
cv_scores_lr = cross_val_score(lr_model, X_train, y_train, cv=min(5, X_train.shape[0]//len(y.unique())), scoring='accuracy')
print(f"  CV Accuracy:       {cv_scores_lr.mean():.4f} (+/- {cv_scores_lr.std():.4f})")


### 6.3 Multinomial Naive Bayes (Baseline)


In [ ]:
print("Training Multinomial Naive Bayes...")

nb_model = MultinomialNB(alpha=0.1)  # Smoothing
nb_model.fit(X_train, y_train)

# Predictions
y_pred_nb_train = nb_model.predict(X_train)
y_pred_nb = nb_model.predict(X_test)

# Evaluation
train_acc_nb = accuracy_score(y_train, y_pred_nb_train)
test_acc_nb = accuracy_score(y_test, y_pred_nb)
precision_nb, recall_nb, f1_nb, _ = precision_recall_fscore_support(y_test, y_pred_nb, average='weighted', zero_division=0)

print(f"\nNaive Bayes Results:")
print(f"  Training Accuracy: {train_acc_nb:.4f}")
print(f"  Test Accuracy:     {test_acc_nb:.4f}")
print(f"  Gap:               {train_acc_nb - test_acc_nb:.4f}")
print(f"  Precision:         {precision_nb:.4f}")
print(f"  Recall:            {recall_nb:.4f}")
print(f"  F1-Score:          {f1_nb:.4f}")

# Cross-validation
cv_scores_nb = cross_val_score(nb_model, X_train, y_train, cv=min(5, X_train.shape[0]//len(y.unique())), scoring='accuracy')
print(f"  CV Accuracy:       {cv_scores_nb.mean():.4f} (+/- {cv_scores_nb.std():.4f})")


## 7. Model Comparison


In [ ]:
# Compare models
comparison = pd.DataFrame({
    'Model': ['Linear SVM', 'Logistic Regression', 'Naive Bayes'],
    'Test Acc': [test_acc, test_acc_lr, test_acc_nb],
    'Train Acc': [train_acc, train_acc_lr, train_acc_nb],
    'Gap': [train_acc - test_acc, train_acc_lr - test_acc_lr, train_acc_nb - test_acc_nb],
    'F1-Score': [f1, f1_lr, f1_nb],
    'CV Acc': [cv_scores.mean(), cv_scores_lr.mean(), cv_scores_nb.mean()]
})

comparison = comparison.sort_values('Test Acc', ascending=False)

print("\n" + "="*80)
print("MODEL COMPARISON")
print("="*80)
print(comparison.to_string(index=False))

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Test accuracy
axes[0].bar(comparison['Model'], comparison['Test Acc'], color=['#2ecc71', '#3498db', '#e74c3c'])
axes[0].set_ylabel('Test Accuracy', fontsize=12)
axes[0].set_title('Model Comparison: Test Accuracy', fontsize=13, fontweight='bold')
axes[0].set_ylim([0, 1])
for i, v in enumerate(comparison['Test Acc']):
    axes[0].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# Plot 2: Overfitting gap
axes[1].bar(comparison['Model'], comparison['Gap'], color=['#2ecc71', '#3498db', '#e74c3c'])
axes[1].set_ylabel('Accuracy Gap (Train - Test)', fontsize=12)
axes[1].set_title('Overfitting Gap', fontsize=13, fontweight='bold')
axes[1].axhline(y=0.05, color='orange', linestyle='--', alpha=0.7, label='Moderate threshold')
axes[1].axhline(y=0.10, color='red', linestyle='--', alpha=0.7, label='High threshold')
axes[1].legend()
for i, v in enumerate(comparison['Gap']):
    axes[1].text(i, v + 0.005, f'{v:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

# Select best
best_idx = comparison.index[0]
best_model_name = comparison.iloc[0]['Model']
best_test_acc = comparison.iloc[0]['Test Acc']

print(f"\nBest Model: {best_model_name} (Test Acc: {best_test_acc:.4f})")


## 8. Detailed Analysis - Per Class Performance


In [ ]:
# Get best model
if best_model_name == 'Linear SVM':
    best_model = svm_model
    y_pred_best = y_pred_svm
elif best_model_name == 'Logistic Regression':
    best_model = lr_model
    y_pred_best = y_pred_lr
else:
    best_model = nb_model
    y_pred_best = y_pred_nb

# Classification report
print(f"\nClassification Report for {best_model_name}:")
print("="*80)
report = classification_report(y_test, y_pred_best, zero_division=0, output_dict=True)
report_df = pd.DataFrame(report).transpose()

# Sort by F1-score to see worst performing classes
report_df = report_df[report_df['support'] > 0].sort_values('f1-score')
print("\nWORST 10 Classes (lowest F1-score):")
print(report_df[['precision', 'recall', 'f1-score', 'support']].head(10).to_string())

print("\n\nBEST 10 Classes (highest F1-score):")
print(report_df[['precision', 'recall', 'f1-score', 'support']].tail(10).to_string())


In [ ]:
# Analyze problem classes (greeting, goodbye, etc.)
problem_classes = ['greeting', 'goodbye', 'pagi', 'siang', 'sore', 'malam']

print("\n" + "="*80)
print("ANALYSIS: Common Interaction Classes (greeting, goodbye, etc.)")
print("="*80)

for cls in problem_classes:
    if cls in report:
        metrics = report[cls]
        print(f"\n{cls}:")
        print(f"  Precision: {metrics['precision']:.3f}")
        print(f"  Recall:    {metrics['recall']:.3f}")
        print(f"  F1-Score:  {metrics['f1-score']:.3f}")
        print(f"  Support:   {int(metrics['support'])} samples")

        if metrics['f1-score'] < 0.5:
            print(f"  ⚠️ CRITICAL: Very low F1-score!")
        elif metrics['f1-score'] < 0.7:
            print(f"  ⚠️ WARNING: Low F1-score")
        else:
            print(f"  ✓ Good performance")


## 9. Confusion Matrix for Problem Classes


In [ ]:
# Focus on interaction classes
interaction_classes = ['greeting', 'goodbye', 'pagi', 'siang', 'sore', 'malam']
interaction_classes = [c for c in interaction_classes if c in y_test.unique()]

if len(interaction_classes) > 0:
    # Filter for these classes only
    mask = y_test.isin(interaction_classes)
    y_test_subset = y_test[mask]
    y_pred_subset = pd.Series(y_pred_best)[mask.values]

    if len(y_test_subset) > 0:
        cm = confusion_matrix(y_test_subset, y_pred_subset, labels=interaction_classes)

        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd',
                    xticklabels=interaction_classes,
                    yticklabels=interaction_classes)
        plt.xlabel('Predicted', fontsize=12)
        plt.ylabel('True', fontsize=12)
        plt.title('Confusion Matrix: Interaction Classes', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

        print("\nNote: Diagonal = correct predictions, off-diagonal = errors")
    else:
        print("No test samples for interaction classes")
else:
    print("Interaction classes not found in test set")


## 10. Root Cause Analysis

**Why greeting/goodbye have low scores?**


In [ ]:
print("="*80)
print("ROOT CAUSE ANALYSIS")
print("="*80)

analysis = """
🔍 PROBLEM: Many Classes (88), Few Samples (~25 each)

1. DATA SCARCITY
   - Only ~25 patterns per intent
   - With 70/30 split: ~17 training, ~8 test samples per class
   - Model has VERY LIMITED examples to learn from

2. CLASS CONFUSION (Pattern Overlap)
   - Generic patterns like "hai", "terima kasih" may appear in multiple intents
   - With few samples, model can't distinguish patterns well

3. MODEL CAPACITY vs DATA
   - 88 classes need significant model capacity
   - But few samples → can't train complex model (overfitting)
   - Trade-off: Simple model (underfits) vs Complex model (overfits)

4. TEST SET TOO SMALL
   - Only ~8 test samples per class
   - A single misclassification = -12.5% accuracy for that class!
   - Metrics become unreliable with such small test sets

5. AUGMENTED DATA WAS WRONG APPROACH
   - Repetitive augmentation → model memorizes exact patterns
   - Doesn't generalize to variations
   - Creates false confidence (100% on seen, 0% on new)

📊 SOLUTIONS:

✅ IMMEDIATE (Already Applied):
   1. Use FULL data (not augmented)
   2. Strong regularization (C=0.1)
   3. Larger test set (30% instead of 20%)
   4. Linear models (not deep networks)
   5. Class balancing

✅ SHORT TERM:
   1. Collect MORE data (50+ patterns per intent)
   2. Use SMARTER augmentation (paraphrasing, not copy)
   3. Merge similar classes (reduce from 88 to ~30)
   4. Hierarchical classification

✅ LONG TERM:
   1. Active learning from user inputs
   2. Transfer learning from pretrained models
   3. Ensemble of specialized models
"""

print(analysis)


## 11. Save Best Model


In [ ]:
# Save best model
with open('chatbot_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)

with open('responses_dict.pkl', 'wb') as f:
    pickle.dump(responses_dict, f)

# Save calibrated SVM for probability estimates
if best_model_name == 'Linear SVM':
    with open('chatbot_model_calibrated.pkl', 'wb') as f:
        pickle.dump(svm_calibrated, f)

print(f"Model saved: {best_model_name}")
print(f"Files: chatbot_model.pkl, tfidf_vectorizer.pkl, responses_dict.pkl")


## 12. Chatbot Interface with Confidence Threshold


In [ ]:
def chatbot_response(user_input, confidence_threshold=0.3):
    # Preprocess
    processed = preprocess_text(user_input)

    if not processed.strip():
        return {
            'response': "Maaf, saya tidak mengerti. Bisa ulangi?",
            'intent': 'unknown',
            'confidence': 0.0
        }

    # Transform
    input_tfidf = tfidf_vectorizer.transform([processed])

    # Predict
    predicted_tag = best_model.predict(input_tfidf)[0]

    # Get confidence
    if hasattr(best_model, 'predict_proba'):
        confidence = max(best_model.predict_proba(input_tfidf)[0])
    elif best_model_name == 'Linear SVM':
        # Use calibrated model for confidence
        try:
            confidence = max(svm_calibrated.predict_proba(input_tfidf)[0])
        except:
            confidence = 0.5
    else:
        confidence = 0.5

    # Confidence check
    if confidence < confidence_threshold:
        return {
            'response': f"Maaf, saya kurang yakin ({confidence:.0%} confidence). Bisa perjelas pertanyaan Anda?",
            'intent': predicted_tag,
            'confidence': confidence
        }

    # Get response
    if predicted_tag in responses_dict:
        response = random.choice(responses_dict[predicted_tag])
    else:
        response = "Maaf, saya tidak punya informasi tentang itu."

    return {
        'response': response,
        'intent': predicted_tag,
        'confidence': confidence
    }

# Test
test_inputs = [
    "halo",
    "selamat pagi",
    "wisata di bantul",
    "terima kasih",
    "kabupaten apa aja di DIY?",
    "kecamatan di sleman"
]

print("\n" + "="*80)
print("CHATBOT TEST")
print("="*80)

for inp in test_inputs:
    result = chatbot_response(inp)
    print(f"\nUser: {inp}")
    print(f"Intent: {result['intent']} (Conf: {result['confidence']:.2f})")
    print(f"Bot: {result['response']}")
    print("-" * 80)


## 13. Summary & Recommendations


In [ ]:
print("\n" + "="*80)
print("PROJECT SUMMARY")
print("="*80)

summary = f"""
📊 DATASET:
   - Source: intents_diy_full.json (original, not augmented)
   - Total patterns: {len(df)}
   - Unique intents: {len(y.unique())}
   - Avg patterns per intent: {len(df) / len(y.unique()):.1f}
   - Training samples: {X_train.shape[0]}
   - Test samples: {X_test.shape[0]}

🎯 BEST MODEL: {best_model_name}
   - Test Accuracy: {best_test_acc:.4f}
   - Precision (weighted): {comparison.loc[comparison['Model'] == best_model_name, 'F1-Score'].values[0]:.4f}
   - Overfitting Gap: {comparison.loc[comparison['Model'] == best_model_name, 'Gap'].values[0]:.4f}

⚠️ CHALLENGES:
   - 88 classes with only ~25 patterns each
   - Some classes (greeting, goodbye) may perform poorly
   - Need MORE data for reliable classification

✅ WHAT WAS DONE RIGHT:
   - Used original data (not augmented repetitions)
   - Strong regularization to prevent overfitting
   - Larger test set (30%) for better evaluation
   - Linear models suitable for small data
   - Confidence threshold to avoid bad predictions

📈 RECOMMENDATIONS:

CRITICAL (Do immediately):
   1. Collect MORE data (target: 100+ patterns per intent)
   2. Merge similar intents (88 → ~30 classes)
   3. Add rule-based fallback for common classes

HIGH PRIORITY:
   4. Implement active learning from user feedback
   5. Use smarter data augmentation (paraphrasing)
   6. Add hierarchical classification

NICE TO HAVE:
   7. A/B test different confidence thresholds
   8. Monitor real-world performance
   9. Regular model retraining with new data

💡 KEY INSIGHT:
   With 88 classes and ~25 samples each, even the best model will struggle.
   This is a DATA problem, not a MODEL problem.
   More data > Better algorithm!
"""

print(summary)

print("\n" + "="*80)
print("NEXT STEPS:")
print("="*80)
print("""
1. Review per-class performance above
2. Identify which classes need more data
3. Collect more patterns for poorly performing classes
4. Consider merging very similar intents
5. Re-train with improved dataset
""")
